In [1]:

import torch
import torch.nn as nn
from torch.utils.cpp_extension import load
import os
import time
import math

curr_path = "."
src_files = [os.path.join(curr_path, 'extension', file) for file in ['cuda_kernel.cu', 'torch_extension.cpp']]
attention = load(
    'binary_proj', 
    src_files, 
    extra_cflags = ['-O3'],
    extra_ldflags = ['-O3'],
    verbose = True
)

import binary_proj

Using /root/.cache/torch_extensions/py311_cu118 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu118/binary_proj/build.ninja...
/root/miniconda3/envs/bpc/lib/python3.11/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module binary_proj...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


ninja: no work to do.


Loading extension module binary_proj...


In [2]:
def smoke_test():
    probe = torch.randint(-15, 15, size = (1, 8, 4, 64), dtype = torch.int, device = "cuda")
    packed_probe = torch.zeros(1, 8, 4, 5, dtype = torch.long, device = "cuda")
    reconstruct = torch.zeros_like(probe)
    binary_proj.format_probe_hash64_fn(probe, packed_probe, True)
    binary_proj.format_probe_hash64_fn(reconstruct, packed_probe, False)
    print((probe - reconstruct).abs().max())

    unpacked = torch.randint(0, 2, size = (1, 8, 1000, 64), dtype = torch.int, device = "cuda") * 2 - 1
    packed = torch.zeros(1, 8, 1000, dtype = torch.long, device = "cuda")
    reconstruct = torch.zeros_like(unpacked)
    binary_proj.format_bits_hash64_fn(unpacked, packed, True)
    binary_proj.format_bits_hash64_fn(reconstruct, packed, False)
    print((unpacked - reconstruct).abs().max())

    unpacked_hashcode = torch.randint(0, 2, size = (1, 8, 8 * 1024, 64), dtype = torch.int, device = "cuda") * 2 - 1
    packed_hashcode = torch.zeros(1, 8, 8 * 1024, dtype = torch.long, device = "cuda")
    binary_proj.format_bits_hash64_fn(unpacked_hashcode, packed_hashcode, True)

    unpacked_probe = torch.randint(-1, 1, size = (1, 8, 4, 64), dtype = torch.int, device = "cuda")
    packed_probe = torch.zeros(1, 8, 4, 5, dtype = torch.long, device = "cuda")
    binary_proj.format_probe_hash64_fn(unpacked_probe, packed_probe, True)

    ref = (unpacked_probe.float() @ unpacked_hashcode.transpose(-1, -2).float()).max(dim = -2).values.to(torch.short)

    scores = torch.zeros(1, 8, 8192, dtype = torch.short, device = "cuda")
    binary_proj.probe_hash64_group4_block256_fn(packed_probe, packed_hashcode, scores)

    counters_cumsum = torch.zeros(1, 8, 32, dtype = torch.int, device = "cuda")
    binary_proj.topk_step1_block512_fn(scores, counters_cumsum)
    print(counters_cumsum[:, :, -1])
    
    indices = torch.zeros(1, 8, 512, dtype = torch.int, device = "cuda")
    binary_proj.topk_step3_block512_fn(scores, counters_cumsum, indices)

    print((scores[-1, -1, indices[-1, -1]] - torch.topk(scores[-1, -1], k = 512).values).abs().max())

    counters = torch.randint(0, 400, size = (1, 8, 32), dtype = torch.int, device = "cuda")
    ref = torch.cumsum(counters, dim = -1).int()
    out = torch.empty_like(counters)
    binary_proj.topk_step2_fn(counters, out)
    print((ref - out).abs().max())

    probe = torch.randn(5, 8, 4, 64, dtype = torch.bfloat16, device = "cuda")
    quant_probe = torch.empty(5, 8, 4, 64, dtype = torch.int, device = "cuda")
    binary_proj.quant_probe_hash64_fn(probe, quant_probe)
    ref = torch.round(probe * 15 / probe.reshape(5, 8, -1).max(dim = -1).values[:, :, None, None]).int()
    print(quant_probe.min(), quant_probe.max())
    print(torch.corrcoef(torch.stack([quant_probe.reshape(-1), ref.reshape(-1)], dim = 0).float()))

smoke_test()

tensor(0, device='cuda:0', dtype=torch.int32)
tensor(0, device='cuda:0', dtype=torch.int32)
tensor([[8192, 8192, 8192, 8192, 8192, 8192, 8192, 8192]], device='cuda:0',
       dtype=torch.int32)
tensor(0, device='cuda:0', dtype=torch.int16)
tensor(0, device='cuda:0', dtype=torch.int32)
tensor(-127, device='cuda:0', dtype=torch.int32) tensor(127, device='cuda:0', dtype=torch.int32)
tensor([[1.0000, 0.9930],
        [0.9930, 1.0000]], device='cuda:0')


In [3]:
def find_binary_projection(A, num_bits, num_iters = 4):
    B, H, L, D = A.shape
    quan_A = torch.empty(B, H, L, num_bits, device = A.device, dtype = A.dtype)
    quan_proj = torch.empty(B, H, num_bits, D, device = A.device, dtype = A.dtype)
    Vh_list = []
    for bit_idx in range(num_bits):
        Vh = torch.randn(B, H, D, device = A.device, dtype = A.dtype)
        for idx in range(num_iters):
            Uq = torch.einsum("bhnd,bhd->bhn", A, Vh).sign()
            Vh = torch.einsum("bhnd,bhn->bhd", A, Uq)
        Vh = Vh / L
        A = A - torch.einsum("bhn,bhd->bhnd", Uq, Vh)
        quan_A[:, :, :, bit_idx] = Uq
        quan_proj[:, :, bit_idx, :] = Vh

    return quan_A, quan_proj

def binary_project_torch(A, quan_proj):
    B, H, L, D = A.shape
    num_bits = quan_proj.shape[-2]
    quan_A = torch.empty(B, H, L, num_bits, device = A.device, dtype = A.dtype)
    for bit_idx in range(num_bits):
        Uq = torch.einsum("bhld,bhd->bhl", A, quan_proj[:, :, bit_idx, :]).sign()
        A = A - torch.einsum("bhl,bhd->bhld", Uq, quan_proj[:, :, bit_idx, :])
        quan_A[:, :, :, bit_idx] = Uq
    return quan_A

def measure_time(method):
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(1000):
        method()
    torch.cuda.synchronize()
    t1 = time.time()
    return t1 - t0

def binary_project_cuda(A, quan_proj):
    B, H, L, D = A.shape
    y = torch.empty(B, H, L, device = A.device, dtype = torch.long)
    binary_proj.binary_project_dim128_hash64_fn(A, quan_proj, y)
    return y

In [4]:


def compute_hashscores(probe, hashcode, T):
    B, H, S, D = probe.shape
    B, H, L = hashcode.shape
    assert S == 4
    assert L % 256 == 0
    assert D == 64
    hashcode = hashcode.contiguous()
    probe = probe.contiguous()
    quant_probe = torch.empty(B, H, S, D, dtype = torch.int, device = probe.device)
    packed_probe = torch.empty(B, H, S, 5, dtype = torch.long, device = probe.device)
    binary_proj.quant_probe_hash64_fn(probe, quant_probe)
    binary_proj.format_probe_hash64_fn(quant_probe, packed_probe, True)

    scores = torch.empty(B, H, L, dtype = torch.short, device = probe.device)
    binary_proj.probe_hash64_group4_block256_fn(packed_probe, hashcode, scores)
    scores[:, :, T:] = -10000
    return scores

def attn(query, key, value, scaling):
    return torch.nn.functional.scaled_dot_product_attention(query, key, value, scale = scaling, enable_gqa = True)

def sparse_attn(query, key, value, scaling, probe, hashcode, k, T):
    B, H, L, D = key.shape
    batch_idxes = torch.arange(B, device = query.device)
    head_idxes = torch.arange(H, device = query.device)
    scores = compute_hashscores(probe, hashcode, T)
    token_idxes = torch.topk(scores, k = k, dim = -1).indices
    
    key = key[batch_idxes[:, None, None], head_idxes[None, :, None], token_idxes, :]
    value = value[batch_idxes[:, None, None], head_idxes[None, :, None], token_idxes, :]
    return torch.nn.functional.scaled_dot_product_attention(query, key, value, scale = scaling, enable_gqa = True)

In [5]:
B, H, L = 1, 8, 512 * 1000
for L in [1, 2, 4, 8, 16, 32 , 64 , 128 ,256 ]:
    L = 1024 * L
    keys = torch.randn(B, H, L, 128, device = "cuda", dtype = torch.bfloat16)
    values = torch.randn(B, H, L, 128, device = "cuda", dtype = torch.bfloat16)
    query = torch.randn(B, H, 4, 128, device = "cuda", dtype = torch.bfloat16)
    
    _, quan_proj = find_binary_projection(keys, 64)
    hashcode = binary_project_cuda(keys, quan_proj)
    probe = query @ quan_proj.transpose(-1, -2)
    
    top2 = int(L*0.02)
    print("L:",L)
    print("top2:",top2)
    print("compute hashscores", measure_time(lambda :compute_hashscores(probe, hashcode, L - 10)))
    print("compute sparse attention", measure_time(lambda :sparse_attn(query, keys, values, 0.1, probe, hashcode, top2, L - 10)))
    print("compute full attention", measure_time(lambda :attn(query, keys, values, 0.1)))

L: 1024
top2: 20
compute hashscores 0.022194862365722656
compute sparse attention 0.14332127571105957
compute full attention 0.016355276107788086
L: 2048
top2: 40
compute hashscores 0.02200007438659668
compute sparse attention 0.1350388526916504
compute full attention 0.017903566360473633
L: 4096
top2: 81
compute hashscores 0.021825790405273438
compute sparse attention 0.13429975509643555
compute full attention 0.025343894958496094
L: 8192
top2: 163
compute hashscores 0.021853923797607422
compute sparse attention 0.13810467720031738
compute full attention 0.04053378105163574
L: 16384
top2: 327
compute hashscores 0.021728038787841797
compute sparse attention 0.13851451873779297
compute full attention 0.06604838371276855
L: 32768
top2: 655
compute hashscores 0.021551847457885742
compute sparse attention 0.17735862731933594
compute full attention 0.12652087211608887
L: 65536
top2: 1310
compute hashscores 0.03346824645996094
compute sparse attention 0.17624187469482422
compute full attenti

In [ ]:
B, H, L = 16, 8, 512 * 1000
for L in [1, 2, 4, 8, 16, 32 , 64 , 128 ]:
    L = 1024 * L
    keys = torch.randn(B, H, L, 128, device = "cuda", dtype = torch.bfloat16)
    values = torch.randn(B, H, L, 128, device = "cuda", dtype = torch.bfloat16)
    query = torch.randn(B, H, 4, 128, device = "cuda", dtype = torch.bfloat16)
    
    _, quan_proj = find_binary_projection(keys, 64)
    hashcode = binary_project_cuda(keys, quan_proj)
    probe = query @ quan_proj.transpose(-1, -2)
    
    top2 = int(L*0.02)
    print("L:",L)
    print("top2:",top2)
    print("compute hashscores", measure_time(lambda :compute_hashscores(probe, hashcode, L - 10)))
    print("compute sparse attention", measure_time(lambda :sparse_attn(query, keys, values, 0.1, probe, hashcode, top2, L - 10)))
    print("compute full attention", measure_time(lambda :attn(query, keys, values, 0.1)))

L: 1024
top2: 20
compute hashscores 0.02167510986328125
compute sparse attention 0.140061616897583
compute full attention 0.12561702728271484
L: 2048
top2: 40
compute hashscores 0.021860599517822266
compute sparse attention 0.13610553741455078
compute full attention 0.2449946403503418
L: 4096
top2: 81
compute hashscores 0.03396749496459961
compute sparse attention 0.14731407165527344
compute full attention 0.48361635208129883
L: 8192
top2: 163
compute hashscores 0.05803656578063965
compute sparse attention 0.22739648818969727
compute full attention 0.9607057571411133
L: 16384
top2: 327
compute hashscores 0.10671305656433105
compute sparse attention 0.37993407249450684
compute full attention 1.8994810581207275
L: 32768
top2: 655
compute hashscores 0.20571064949035645
compute sparse attention 0.6651017665863037
compute full attention 3.7920327186584473
L: 65536
top2: 1310
compute hashscores 0.4002342224121094
compute sparse attention 1.2273495197296143
compute full attention 7.5764565467

: 